# 公开混凝土图片的20图对照验证

第五章GPU提示v2开发回归：20张227×227裁剪图，来源正/负类各10张，不能代表20个独立现场。三张用于提示诊断，本组不作为独立盲测。手机和GPU使用相同图像哈希，解码与视觉预处理配置不同，时间不作直接等效比较。来源、许可及边界歧义见数据子集README。

## 1 固定环境和数据
使用此前已准备的Part5内核，不重新升级PyTorch。

In [ ]:
from pathlib import Path
import sys, subprocess, json
candidates=[Path.cwd(), Path.cwd().parent, Path.cwd()/"minicpm-purplebook-draft"]
ROOT=next((p.resolve() for p in candidates if (p/"examples/task_contract.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请先上传并解压完整实践包，再在项目目录打开Notebook")
sys.path.insert(0,str(ROOT/"examples"))
def run(args):
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True)
print("Project:",ROOT)

import hashlib, torch, transformers
assert transformers.__version__ == "5.7.0"
assert torch.cuda.is_available()
MODEL=ROOT/"models/minicpm-v-4.6"
manifest=json.loads((MODEL/"download-manifest.json").read_text())
for item in manifest["files"]:
    path=MODEL/item["path"]
    with path.open("rb") as f: actual=hashlib.file_digest(f,"sha256").hexdigest()
    assert actual==item["sha256"], item["path"]
print("Model files verified:",len(manifest["files"]))
DATA=ROOT/"data/inspection-validation/public-cracks/manifest.json"
assert len(json.loads(DATA.read_text())["images"])==20


## 2 16x 连续运行
单次模型加载，保存原始输出、结构校验、逐图时间与PyTorch显存峰值。

In [ ]:
run([sys.executable,ROOT/"examples/vision_batch.py","--model",MODEL,"--manifest",DATA,
     "--downsample","16x","--out",ROOT/"results/benchmark-gpu-v2-16x.json"])


## 3 同图4x对照
预处理和生成两处参数一致；每种模式单独加载一次模型。

In [ ]:
run([sys.executable,ROOT/"examples/vision_batch.py","--model",MODEL,"--manifest",DATA,
     "--downsample","4x","--out",ROOT/"results/benchmark-gpu-v2-4x.json"])


## 4 检查结果
结构合格率单独统计。对象、裂纹描述和无依据断言需按保存的标注逐图复核。

In [ ]:
for mode in ["16x","4x"]:
    report=json.loads((ROOT/f"results/benchmark-gpu-v2-{mode}.json").read_text())
    assert report["status"]=="completed" and len(report["rows"])==20
    print(mode,"schema",sum(r["schema_valid"] for r in report["rows"]),"/20",
          "allocated MiB",report["cuda_peak_allocated_bytes"]/1024**2)
